# ETL Pipeline — Riesgo de Desastres Naturales en México

**Pipeline completo Extract → Transform → Load** sobre Amazon Aurora PostgreSQL.

| Etapa | Descripción |
|---|---|
| **Extract** | Lectura de 3 tablas staging (`stg_declaratorias_desastre`, `stg_declaratorias_emergencia`, `stg_proyectos_prevencion`) |
| **Transform** | Normalización de estados, limpieza de montos/años, deduplicación, agregación por grano |
| **Load** | Carga de `dim_estado`, `dim_tiempo`, `dim_fenomeno`, `dim_programa_prevencion` y `fact_riesgo` |
| **Validate** | Checks post-carga: FK nulas, valores negativos, resumen de totales |

> **Prerequisito:** las tablas `stg_*` deben estar cargadas y el DDL del star schema ejecutado (`01_schema_ddl.sql`).

## 0. Instalación de dependencias

Ejecuta esta celda solo la primera vez.

In [1]:
# Solo es necesario ejecutar esta celda una vez
%pip install pandas sqlalchemy psycopg2-binary tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Imports y configuración de logging

In [2]:
import logging
import re
import unicodedata
import warnings

import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, Numeric, Text
from tqdm import tqdm

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    force=True,
)
logger = logging.getLogger('etl_desastres')
print('✓ Imports OK')

✓ Imports OK


## 2. Configuración de conexión a Aurora PostgreSQL

Reemplaza los valores con los de tu cluster del Tema 01.

In [3]:
# ─── EDITA ESTOS VALORES ──────────────────────────────────────────────────
AURORA_HOST     = 'aurora-mod4.cluster-cxjugulcqnkv.us-east-1.rds.amazonaws.com'
AURORA_PORT     = 5432
AURORA_DATABASE = 'northwind'
AURORA_USER     = 'postgres'
AURORA_PASSWORD = 'Metzme1703'
# ──────────────────────────────────────────────────────────────────────────

CONNECTION_STRING = (
    f'postgresql+psycopg2://{AURORA_USER}:{AURORA_PASSWORD}'
    f'@{AURORA_HOST}:{AURORA_PORT}/{AURORA_DATABASE}'
)

engine = create_engine(CONNECTION_STRING, pool_pre_ping=True)

# Test de conexión
with engine.connect() as conn:
    result = conn.execute(text('SELECT current_database(), current_user')).fetchone()
    print(f'✓ Conectado a: {result[0]}  |  Usuario: {result[1]}')

✓ Conectado a: northwind  |  Usuario: postgres


## 3. Catálogo de normalización de entidades federativas

Mapea 45+ variantes encontradas en los datos fuente a los **32 nombres oficiales**.
Por ejemplo: `'CDMX'`, `'Distrito Federal'`, `'D.F.'` → `'Ciudad de México'`.

In [4]:
NORM_ESTADOS: dict[str, str] = {
    'AGUASCALIENTES': 'Aguascalientes',
    'BAJA CALIFORNIA': 'Baja California',
    'BAJA CALIFORNIA SUR': 'Baja California Sur',
    'CAMPECHE': 'Campeche',
    'CHIAPAS': 'Chiapas',
    'CHIHUAHUA': 'Chihuahua',
    'CIUDAD DE MEXICO': 'Ciudad de México',
    'CIUDAD DE MÉXICO': 'Ciudad de México',
    'CDMX': 'Ciudad de México',
    'DISTRITO FEDERAL': 'Ciudad de México',
    'D.F.': 'Ciudad de México',
    'COAHUILA': 'Coahuila',
    'COAHUILA DE ZARAGOZA': 'Coahuila',
    'COLIMA': 'Colima',
    'DURANGO': 'Durango',
    'GUANAJUATO': 'Guanajuato',
    'GUERRERO': 'Guerrero',
    'HIDALGO': 'Hidalgo',
    'JALISCO': 'Jalisco',
    'MEXICO': 'Estado de México',
    'ESTADO DE MEXICO': 'Estado de México',
    'ESTADO DE MÉXICO': 'Estado de México',
    'EDO. DE MEX.': 'Estado de México',
    'MICHOACAN': 'Michoacán',
    'MICHOACÁN': 'Michoacán',
    'MICHOACAN DE OCAMPO': 'Michoacán',
    'MORELOS': 'Morelos',
    'NAYARIT': 'Nayarit',
    'NUEVO LEON': 'Nuevo León',
    'NUEVO LEÓN': 'Nuevo León',
    'OAXACA': 'Oaxaca',
    'PUEBLA': 'Puebla',
    'QUERETARO': 'Querétaro',
    'QUERÉTARO': 'Querétaro',
    'QUINTANA ROO': 'Quintana Roo',
    'SAN LUIS POTOSI': 'San Luis Potosí',
    'SAN LUIS POTOSÍ': 'San Luis Potosí',
    'SINALOA': 'Sinaloa',
    'SONORA': 'Sonora',
    'TABASCO': 'Tabasco',
    'TAMAULIPAS': 'Tamaulipas',
    'TLAXCALA': 'Tlaxcala',
    'VERACRUZ': 'Veracruz',
    'VERACRUZ DE IGNACIO DE LA LLAVE': 'Veracruz',
    'YUCATAN': 'Yucatán',
    'YUCATÁN': 'Yucatán',
    'ZACATECAS': 'Zacatecas',
}

ESTADOS_VALIDOS: set[str] = set(NORM_ESTADOS.values())

print(f'✓ Catálogo cargado: {len(NORM_ESTADOS)} variantes → {len(ESTADOS_VALIDOS)} estados oficiales')

✓ Catálogo cargado: 47 variantes → 32 estados oficiales


## 4. Funciones de limpieza (helpers)

Cuatro funciones reutilizables que aplican las transformaciones atómicas sobre cada columna.

In [5]:
def normalizar_estado(nombre: str | None) -> str | None:
    """
    Normaliza el nombre de una entidad federativa a su forma oficial.
    Pasos: strip → upper → quitar acentos → buscar en NORM_ESTADOS.
    """
    if not nombre or pd.isna(nombre):
        return None
    limpio = str(nombre).strip().upper()
    sin_acentos = ''.join(
        c for c in unicodedata.normalize('NFD', limpio)
        if unicodedata.category(c) != 'Mn'
    )
    return NORM_ESTADOS.get(sin_acentos) or NORM_ESTADOS.get(limpio)


def limpiar_monto(valor: str | None) -> float | None:
    """
    Convierte cadenas de montos a float.
    Acepta: '$1,200,000.50', '1200000', '1.2e6', 'N/D', None.
    """
    if valor is None or pd.isna(valor):
        return None
    texto = str(valor).strip().replace('$', '').replace(',', '').replace(' ', '')
    if texto in ('', 'N/D', 'ND', 'NaN', '0.00'):
        return None
    try:
        return float(texto)
    except ValueError:
        return None


def limpiar_entero(valor: str | None) -> int | None:
    """Convierte cadenas a int, tolerando comas de miles y sufijos de texto."""
    if valor is None or pd.isna(valor):
        return None
    texto = re.sub(r'[^\d]', '', str(valor))
    return int(texto) if texto else None


def limpiar_anio(valor: str | None) -> int | None:
    """Extrae un año de 4 dígitos válido (2000-2030)."""
    if valor is None or pd.isna(valor):
        return None
    m = re.search(r'\b(20[0-2]\d)\b', str(valor))
    return int(m.group(1)) if m else None


def normalizar_fenomeno(valor: str | None) -> str | None:
    """Normaliza el tipo de fenómeno: strip + title-case."""
    if not valor or pd.isna(valor):
        return None
    return str(valor).strip().title()


# ── Prueba rápida de los helpers ──────────────────────────────────────────────
assert normalizar_estado('CDMX')                       == 'Ciudad de México'
assert normalizar_estado('veracruz de ignacio de la llave'.upper()) == 'Veracruz'
assert limpiar_monto('$1,200,000.50')                  == 1_200_000.50
assert limpiar_monto('N/D')                            is None
assert limpiar_anio('Declaratoria 2019')               == 2019
assert limpiar_entero('1,500 personas')                == 1500
print('✓ Todos los helpers pasan las pruebas unitarias')

✓ Todos los helpers pasan las pruebas unitarias


---
## 5. EXTRACT — Lectura desde staging

Lee las tres tablas staging directamente desde Aurora. Todas las columnas llegan como `object` (TEXT) — el cast ocurre en la etapa Transform.

In [6]:
logger.info('EXTRACT: leyendo tablas staging...')

df_desastres_raw = pd.read_sql(
    'SELECT * FROM desastres.stg_declaratorias_desastre',
    engine,
)

df_emergencias_raw = pd.read_sql(
    'SELECT * FROM desastres.stg_declaratorias_emergencia',
    engine,
)

df_proyectos_raw = pd.read_sql(
    'SELECT * FROM desastres.stg_proyectos_prevencion',
    engine,
)

print(f'stg_declaratorias_desastre  : {df_desastres_raw.shape[0]:>6,} filas × {df_desastres_raw.shape[1]} columnas')
print(f'stg_declaratorias_emergencia: {df_emergencias_raw.shape[0]:>6,} filas × {df_emergencias_raw.shape[1]} columnas')
print(f'stg_proyectos_prevencion    : {df_proyectos_raw.shape[0]:>6,} filas × {df_proyectos_raw.shape[1]} columnas')

2026-06-10 21:10:30,130 [INFO] EXTRACT: leyendo tablas staging...


stg_declaratorias_desastre  :    122 filas × 18 columnas
stg_declaratorias_emergencia:    194 filas × 84 columnas
stg_proyectos_prevencion    :     34 filas × 22 columnas


### Preview — `stg_declaratorias_desastre`

In [7]:
display(df_desastres_raw.head(3))
print('Tipos:', df_desastres_raw.dtypes.unique())

,no,entidad_federativa,anio,evento,tipo_fenomeno,instancia_corroboradora,documento_corroboracion,nombre_municipios_corroborados,numero_municipios_corroborados,instalacion_ced,sectores_afectados,fecha_publicacion_dof,publicacion_dof,entrega_resultados_ced,limite_para_entregar_diagnostico,oficio_entrega,observaciones,ultima_fecha_actualizacion
0,1,Jalisco,2019,Inundación fluvial el dos de junio de 2019,Hidrometeorológico,CONAGUA,B00.8.02.-085,San Gabriel,1,07/06/2019,"Educativo, Forestal y de Viveros, Hidráulico, ...",2019-06-13,https://www.dof.gob.mx/nota_detalle.php?codigo...,2019-07-04,2019-07-15,SSPC/SPPPCCP/CNPC/DGGR/0007/2019; SSPC/SPPPCCP...,sin dato,2020
1,2,Tamaulipas,2019,Inundación fluvial y pluvial el 24 de junio de...,Hidrometeorológico,CONAGUA,B00.8.02.-117,Reynosa,1,28/06/2019,Educativo e Hidráulico,2019-07-04,https://www.dof.gob.mx/nota_detalle.php?codigo...,2019-07-25,2019-08-05,SSPC/SPPPCCP/CNPC/DGGR/0142/2019; SSPC/SPPPCCP...,sin dato,2020
2,3,Chiapas,2019,Lluvia severa ocurrida el 17 de agosto de 2019,Hidrometeorológico,CONAGUA,B00.8.-586,Huixtla,1,26/08/2019,"Carretero, Educativo, Hidráulico, Pesquero y A...",2019-08-30,https://www.dof.gob.mx/nota_detalle.php?codigo...,2019-09-06,2019-09-18,SSPC/SPPPCCP/CNPC/DGGR/0713/2019; SSPC/SPPPCCP...,sin dato,2020


Tipos: [dtype('int64') <StringDtype(storage='python', na_value=nan)>]


### Preview — `stg_declaratorias_emergencia`

In [8]:
display(df_emergencias_raw.head(3))
print('Columnas:', df_emergencias_raw.columns.tolist())

,no,entidad_federativa,anio_ocurrencia_evento,boletin_prensa,fecha_emision_boletin_prensa,tipo_fenomeno,amenaza_natural,instancia_corroboradora,nombre_municipios_corroborados,numero_municipios_corroborados,...,fletes,servicio_letrinas,servicio_regaderas,arrendamiento_montacargas,arrendamiento_generadores_energia,lote_insumos_para_salud,costo_fuerzas_armadas,costo_estimado_sspc,costo_total_declaratoria,observaciones
0,1.0,Veracruz,2018,006/18,07-dic-18,Hidrometeorológico,Lluvia severa e inundación pluvial del 04 al 0...,CONAGUA,"Angel R. Cabada, Catemaco, Hueyapan de Ocampo,...",6,...,0,0,0,0,0,0,$-,"$12,274,813.19","$12,274,813.19",
1,2.0,Veracruz,2018,009/18,13-dic-18,Hidrometeorológico,Lluvia severa el 5 de diciembre de 2018,CONAGUA,Lerdo de Tejada y Playa Vicente,2,...,0,0,0,0,0,0,$-,"$1,954,087.62","$1,954,087.62",
2,3.0,Chihuahua,2018,014/18,21-dic-18,Hidrometeorológico,Helada severa a partir del día 21 de diciembre...,CONAGUA,"Chínipas, Guazapares, Maguarichi, Temósachic, ...",15,...,0,0,0,0,0,0,$-,"$16,181,232.84","$16,181,232.84",


Columnas: ['no', 'entidad_federativa', 'anio_ocurrencia_evento', 'boletin_prensa', 'fecha_emision_boletin_prensa', 'tipo_fenomeno', 'amenaza_natural', 'instancia_corroboradora', 'nombre_municipios_corroborados', 'numero_municipios_corroborados', 'oficio_poblacion_afectada', 'poblacion_afectada', 'poblacion_atendida', 'fecha_publicacion_declaratoria_dof', 'publicacion_dof', 'instancia_que_elaboro_informe_utilizacion_insumos', 'boletin_prensa_aviso_termino_emergencia', 'fecha_emision_boletin_prensa_termino_situacion_emergencia', 'fecha_publicacion_aviso_termino_declaratoria_dof', 'publicacion_termino_dof', 'total_insumos_autorizados', 'despensas', 'despensa_sobrevivencia', 'despensa_mantenimiento', 'alimentos_consumo_inmediato', 'fruta_para_poblacion', 'alimento_granel', 'cobertor_a', 'cobertor_b', 'colchoneta', 'hamaca', 'lamina_tipo_a', 'lamina_tipo_b', 'lamina_tipo_c', 'palma_sintetica', 'palma_natural', 'litros_agua', 'kit_limpieza', 'kit_aseo_personal', 'costales', 'saco_absorbente'

### Preview — `stg_proyectos_prevencion`

In [9]:
display(df_proyectos_raw.head(3))

,consecutivo,entidad_federativa_solicitante,instancia_publica_orden_federal_solicitante,nombre_proyecto,anio_autorizacion,instancia_ejecutora,tipo_proyecto,tipo_fenomeno_01,tipo_fenomeno_2,tipo_fenomeno_3,...,mujeres,hombres,poblacion_indigena,anio_finalizacion,proyecto_estrategico,recurso_fondo_preventivo_federal,recurso_coparticipacion_estatal,costo_total_mxn_peso,pagina_web_instancia_autorizada,ultima_fecha_actualizacion
0,1,sin dato,Comision Nacional del Agua,Sistema de Alertamiento a tiempo real para la ...,2013,Organismo de Cuenca Golfo Norte,Proyecto Preventivo,"Hidrometeorologico (ciclon tropical, lluvias e...",sin dato,sin dato,...,SD,SD,SD,sin dato,sin dato,9320624.4,5018809.6,14339434.0,Libro blanco: http://dggr.cenapred.unam.mx/por...,2020
1,2,Quintana Roo,sin dato,Atlas Estatal de Riesgo del Estado de Quintana...,2013,Secretaria de Finanzas y Planeacion de Quintan...,Proyecto Preventivo,"Geologico (tsunamis, Karstificacion y Cavidades)","hidrometeorologico (ciclon tropical, lluvias e...","Incendios forestales, riesgos quimicos-tecnolo...",...,SD,SD,SD,2021.0,sin dato,47962285.31,11990571.33,59952856.64,http://www.sefiplan.qroo.gob.mx/site/pagina.ph...,2021
2,3,sin dato,Universidad Autonoma de Mexico,Desarrollo de herramientas para simulacion de ...,2013,Instituto de Geofisica,Proyecto Preventivo Estrategico,Geologico (vulcanismo),sin dato,sin dato,...,SD,SD,SD,2021.0,SI,5102522.6,0.0,5102522.6,Libro blanco: http://dggr.cenapred.unam.mx/por...,2021


---
## 6. TRANSFORM — `stg_declaratorias_desastre`

Limpieza aplicada:
- **Entidad federativa** → diccionario de normalización + filtro de estados válidos.
- **Año** → regex que extrae 4 dígitos (2000-2030).
- **Tipo fenómeno** → title-case + strip.
- Descarte de registros con campos clave nulos.

In [10]:
def transform_desastres(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['entidad_federativa'] = df['entidad_federativa'].apply(normalizar_estado)
    df['anio']               = df['anio'].apply(limpiar_anio)
    df['tipo_fenomeno']      = df['tipo_fenomeno'].apply(normalizar_fenomeno)

    before = len(df)
    df = df.dropna(subset=['entidad_federativa', 'anio', 'tipo_fenomeno'])
    df = df[df['entidad_federativa'].isin(ESTADOS_VALIDOS)]

    print(f'Registros desastres : {before:,} → {len(df):,}  '
          f'(descartados: {before - len(df):,})')
    return df[['entidad_federativa', 'anio', 'tipo_fenomeno']]


df_des = transform_desastres(df_desastres_raw)
display(df_des.head(5))

Registros desastres : 122 → 122  (descartados: 0)


,entidad_federativa,anio,tipo_fenomeno
0,Jalisco,2019,Hidrometeorológico
1,Tamaulipas,2019,Hidrometeorológico
2,Chiapas,2019,Hidrometeorológico
3,Veracruz,2019,Geológico
4,Nuevo León,2019,Hidrometeorológico


In [11]:
# Distribución de registros por estado (top 10)
display(
    df_des.groupby('entidad_federativa')
          .size()
          .sort_values(ascending=False)
          .head(10)
          .rename('declaratorias')
          .to_frame()
)

,declaratorias
entidad_federativa,
Chiapas,23
Veracruz,19
Oaxaca,14
Baja California Sur,8
Guerrero,6
Jalisco,6
Colima,5
Tabasco,5
Durango,4


---
## 7. TRANSFORM — `stg_declaratorias_emergencia`

Limpieza adicional:
- **Municipios corroborados** → `limpiar_entero` (elimina comas de miles).
- **Población afectada** → `limpiar_monto` (maneja `'N/D'`, `'$'`, comas).
- **tipo_fenomeno** puede venir como `amenaza_natural` — el código lo detecta automáticamente.

In [12]:
def transform_emergencias(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.rename(columns={'anio_ocurrencia_evento': 'anio'})

    df['entidad_federativa'] = df['entidad_federativa'].apply(normalizar_estado)
    df['anio']               = df['anio'].apply(limpiar_anio)

    if 'tipo_fenomeno' in df.columns:
        df['tipo_fenomeno'] = df['tipo_fenomeno'].apply(normalizar_fenomeno)
    elif 'amenaza_natural' in df.columns:
        df['tipo_fenomeno'] = df['amenaza_natural'].apply(normalizar_fenomeno)
    else:
        df['tipo_fenomeno'] = None

    df['numero_municipios_corroborados'] = df['numero_municipios_corroborados'].apply(limpiar_entero)
    df['poblacion_afectada']             = df['poblacion_afectada'].apply(limpiar_monto)
    df['costo_total_declaratoria']       = df['costo_total_declaratoria'].apply(limpiar_monto)

    before = len(df)
    df = df.dropna(subset=['entidad_federativa', 'anio', 'tipo_fenomeno'])
    df = df[df['entidad_federativa'].isin(ESTADOS_VALIDOS)]

    print(f'Registros emergencias: {before:,} → {len(df):,}  '
          f'(descartados: {before - len(df):,})')
    return df[[
        'entidad_federativa', 'anio', 'tipo_fenomeno',
        'numero_municipios_corroborados', 'poblacion_afectada',
    ]]


df_eme = transform_emergencias(df_emergencias_raw)
display(df_eme.head(5))

Registros emergencias: 194 → 190  (descartados: 4)


,entidad_federativa,anio,tipo_fenomeno,numero_municipios_corroborados,poblacion_afectada
0,Veracruz,2018.0,Hidrometeorológico,6,13732.0
1,Veracruz,2018.0,Hidrometeorológico,2,4042.0
2,Chihuahua,2018.0,Hidrometeorológico,15,24344.0
3,Durango,2018.0,Hidrometeorológico,8,14773.0
4,Estado de México,2018.0,Hidrometeorológico,7,42777.0


In [13]:
# Resumen estadístico de columnas numéricas
display(df_eme[['numero_municipios_corroborados', 'poblacion_afectada']].describe())

,numero_municipios_corroborados,poblacion_afectada
count,190.000000,190.000000
mean,7.742105,16792.705263
std,10.527958,63345.608591
min,1.000000,0.000000
25%,2.000000,1188.250000
50%,4.000000,4027.500000
75%,8.000000,12988.000000
max,72.000000,812000.000000


---
## 8. TRANSFORM — `stg_proyectos_prevencion`

Cambios de nombre de columnas:
- `entidad_federativa_solicitante` → `entidad_federativa`
- `anio_autorizacion` → `anio`
- `tipo_fenomeno_01` → `tipo_fenomeno`

In [14]:
def transform_proyectos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.rename(columns={
        'entidad_federativa_solicitante': 'entidad_federativa',
        'anio_autorizacion': 'anio',
        'tipo_fenomeno_01': 'tipo_fenomeno',
    })

    df['entidad_federativa']   = df['entidad_federativa'].apply(normalizar_estado)
    df['anio']                 = df['anio'].apply(limpiar_anio)
    df['tipo_fenomeno']        = df['tipo_fenomeno'].apply(normalizar_fenomeno)
    df['costo_total_mxn_peso'] = df['costo_total_mxn_peso'].apply(limpiar_monto)
    df['personas_beneficiadas']= df['personas_beneficiadas'].apply(limpiar_entero)

    for col in ('estatus', 'tipo_proyecto'):
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()
    if 'nombre_proyecto' in df.columns:
        df['nombre_proyecto'] = df['nombre_proyecto'].str.strip()

    before = len(df)
    df = df.dropna(subset=['entidad_federativa', 'anio'])
    df = df[df['entidad_federativa'].isin(ESTADOS_VALIDOS)]

    print(f'Registros proyectos  : {before:,} → {len(df):,}  '
          f'(descartados: {before - len(df):,})')
    return df[[
        'entidad_federativa', 'anio', 'tipo_fenomeno',
        'nombre_proyecto', 'tipo_proyecto', 'estatus',
        'personas_beneficiadas', 'costo_total_mxn_peso',
    ]]


df_pro = transform_proyectos(df_proyectos_raw)
display(df_pro.head(5))

Registros proyectos  : 34 → 6  (descartados: 28)


,entidad_federativa,anio,tipo_fenomeno,nombre_proyecto,tipo_proyecto,estatus,personas_beneficiadas,costo_total_mxn_peso
1,Quintana Roo,2013,"Geologico (Tsunamis, Karstificacion Y Cavidades)",Atlas Estatal de Riesgo del Estado de Quintana...,Proyecto Preventivo,Finalizado,5000.0,59952856.64
5,Chiapas,2014,"Geologico (Sismo, Inestabilidad De Laderas)",Sistema de Multi Alerta y Comunicacion Masiva ...,Proyecto Preventivo,En Ejecucion,2000000.0,31300375.36
6,Sinaloa,2014,"Geologico (Sismo, Inestabilidad De Laderas Y T...","Atlas de Riesgos, Nivel de Peligros y Levantam...",Proyecto Preventivo,Desistimiento,0.0,0.00
7,Ciudad de México,2015,"Geologico (Sismo, Vulcanismo, Hundimiento, Sub...",Desarrollo del Sistema integrador del Atlas de...,Proyecto Preventivo,Finalizado,8000000.0,29900000.00
25,Yucatán,2022,Geologico (Hundimiento),Actualizacion del Atlas de Peligros de Yucatan...,Proyecto Preventivo A,Pendiente De Inicio,2320898.0,10429503.00


In [15]:
# Top estados por inversión en proyectos preventivos
display(
    df_pro.groupby('entidad_federativa')['costo_total_mxn_peso']
          .sum()
          .sort_values(ascending=False)
          .head(10)
          .rename('inversion_total_mxn')
          .to_frame()
          .style.format('{:,.0f}')
)

,inversion_total_mxn
entidad_federativa,
Quintana Roo,"59,952,857"
Chiapas,"31,300,375"
Ciudad de México,"29,900,000"
Campeche,"13,332,086"
Yucatán,"10,429,503"
Sinaloa,0


---
## 9. BUILD FACT — Construcción de `fact_riesgo`

**Grano:** una fila por `(entidad_federativa × anio × tipo_fenomeno)`.

Las tres fuentes se agregan por separado y se unen con **outer join** para no perder
combinaciones que solo existen en una fuente (ej.: un estado con emergencias pero sin proyectos).

In [16]:
def build_fact(df_des: pd.DataFrame,
               df_eme: pd.DataFrame,
               df_pro: pd.DataFrame) -> pd.DataFrame:

    KEY = ['entidad_federativa', 'anio', 'tipo_fenomeno']

    # Desastres — conteo de declaratorias
    agg_des = (
        df_des.groupby(KEY, as_index=False)
              .size()
              .rename(columns={'size': 'total_desastres'})
    )

    # Emergencias — conteo + métricas de impacto
    agg_eme = (
        df_eme.groupby(KEY, as_index=False)
              .agg(
                  total_emergencias    = ('tipo_fenomeno', 'size'),
                  municipios_afectados = ('numero_municipios_corroborados', 'sum'),
                  poblacion_afectada   = ('poblacion_afectada', 'sum'),
              )
    )

    # Proyectos — inversión preventiva
    df_pro_key = df_pro.dropna(subset=['tipo_fenomeno'])
    agg_pro = (
        df_pro_key.groupby(KEY, as_index=False)
                  .agg(inversion_prevencion=('costo_total_mxn_peso', 'sum'))
    )

    # Full outer join
    fact = (
        agg_des
        .merge(agg_eme, on=KEY, how='outer')
        .merge(agg_pro, on=KEY, how='outer')
    )

    # Rellenar NaN
    fact['total_desastres']      = fact['total_desastres'].fillna(0).astype(int)
    fact['total_emergencias']    = fact['total_emergencias'].fillna(0).astype(int)
    fact['municipios_afectados'] = fact['municipios_afectados'].fillna(0).astype(int)
    fact['poblacion_afectada']   = fact['poblacion_afectada'].fillna(0)
    fact['inversion_prevencion'] = fact['inversion_prevencion'].fillna(0)

    return fact


fact = build_fact(df_des, df_eme, df_pro)
print(f'fact_riesgo construida: {len(fact):,} filas × {fact.shape[1]} columnas')
display(fact.head(8))

fact_riesgo construida: 109 filas × 8 columnas


,entidad_federativa,anio,tipo_fenomeno,total_desastres,total_emergencias,municipios_afectados,poblacion_afectada,inversion_prevencion
0,Baja California,2019.0,Incendio Forestal,1,1,2,1500.0,0.0
1,Baja California,2022.0,Hidrometeorológico,1,0,0,0.0,0.0
2,Baja California,2023.0,Hidrometeorológico,1,0,0,0.0,0.0
3,Baja California Sur,2019.0,Hidrometeorológico,2,4,9,13203.0,0.0
4,Baja California Sur,2020.0,Hidrometeorológico,1,2,3,5060.0,0.0
5,Baja California Sur,2021.0,Hidrometeorológico,1,1,2,8037.0,0.0
6,Baja California Sur,2022.0,Hidrometeorológico,2,0,0,0.0,0.0
7,Baja California Sur,2023.0,Hidrometeorológico,2,1,1,12000.0,0.0


In [17]:
# Resumen de la fact antes de cargar
print('── Totales agregados ──────────────────────')
print(f"  total_desastres     : {fact['total_desastres'].sum():>10,}")
print(f"  total_emergencias   : {fact['total_emergencias'].sum():>10,}")
print(f"  municipios_afect.   : {fact['municipios_afectados'].sum():>10,}")
print(f"  poblacion_afectada  : {fact['poblacion_afectada'].sum():>10,.0f}")
print(f"  inversion_prevencion: {fact['inversion_prevencion'].sum():>10,.0f}")

── Totales agregados ──────────────────────
  total_desastres     :        122
  total_emergencias   :        190
  municipios_afect.   :      1,471
  poblacion_afectada  :  3,190,614
  inversion_prevencion: 144,914,821


---
## 10. LOAD — Dimensiones

Las dimensiones se cargan con `ON CONFLICT ... DO NOTHING` (upsert seguro).
El orden importa: las dims deben existir antes de cargar la fact.

In [18]:
def upsert_dim_estado(estados: list[str], engine) -> pd.DataFrame:
    with engine.begin() as conn:
        for estado in sorted(set(estados)):
            conn.execute(text("""
                INSERT INTO desastres.dim_estado (entidad_federativa)
                VALUES (:estado)
                ON CONFLICT (entidad_federativa) DO NOTHING
            """), {'estado': estado})
    dim = pd.read_sql('SELECT id_estado AS "Id_estado", entidad_federativa FROM desastres.dim_estado', engine)
    print(f'✓ dim_estado         : {len(dim):,} entidades')
    return dim


def upsert_dim_tiempo(anios: list[int], engine) -> pd.DataFrame:
    with engine.begin() as conn:
        for anio in sorted(set(int(a) for a in anios if pd.notna(a))):
            conn.execute(text("""
                INSERT INTO desastres.dim_tiempo (anio)
                VALUES (:anio)
                ON CONFLICT (anio) DO NOTHING
            """), {'anio': anio})
    dim = pd.read_sql('SELECT id_tiempo AS "Id_tiempo", anio FROM desastres.dim_tiempo', engine)
    print(f'✓ dim_tiempo         : {len(dim):,} años  ({dim.anio.min()} - {dim.anio.max()})')
    return dim


def upsert_dim_fenomeno(fenomenos: list[str], engine) -> pd.DataFrame:
    with engine.begin() as conn:
        for fenomeno in sorted(set(f for f in fenomenos if pd.notna(f))):
            conn.execute(text("""
                INSERT INTO desastres.dim_fenomeno (tipo_fenomeno)
                VALUES (:fenomeno)
                ON CONFLICT DO NOTHING
            """), {'fenomeno': fenomeno})
    dim = pd.read_sql('SELECT id_fenomeno AS "Id_fenomeno", tipo_fenomeno FROM desastres.dim_fenomeno', engine)
    print(f'✓ dim_fenomeno       : {len(dim):,} fenómenos')
    return dim


def load_dim_programa(df_pro: pd.DataFrame, engine):
    cols = ['nombre_proyecto', 'tipo_proyecto', 'estatus']
    df_unique = (
        df_pro[cols]
        .drop_duplicates(subset=['nombre_proyecto'])
        .dropna(subset=['nombre_proyecto'])
    )
    df_unique.to_sql(
        'dim_programa_prevencion', engine, schema='desastres',
        if_exists='append', index=False, method='multi',
        dtype={'nombre_proyecto': Text(), 'tipo_proyecto': Text(), 'estatus': Text()},
    )
    print(f'✓ dim_programa_prev. : {len(df_unique):,} proyectos únicos')


# ── Ejecutar ──────────────────────────────────────────────────────────────────
todos_estados   = list(fact['entidad_federativa'].dropna().unique())
todos_anios     = list(fact['anio'].dropna().unique())
todos_fenomenos = list(fact['tipo_fenomeno'].dropna().unique())

dim_estado   = upsert_dim_estado(todos_estados, engine)
dim_tiempo   = upsert_dim_tiempo(todos_anios, engine)
dim_fenomeno = upsert_dim_fenomeno(todos_fenomenos, engine)
load_dim_programa(df_pro, engine)

✓ dim_estado         : 28 entidades
✓ dim_tiempo         : 10 años  (2013 - 2024)
✓ dim_fenomeno       : 45 fenómenos
✓ dim_programa_prev. : 6 proyectos únicos


In [19]:
# Preview de las dims cargadas
print('── dim_estado (primeros 5) ─────────────────')
display(dim_estado.head())
print('── dim_tiempo ──────────────────────────────')
display(dim_tiempo.head())
print('── dim_fenomeno ────────────────────────────')
display(dim_fenomeno.head())

── dim_estado (primeros 5) ─────────────────


,Id_estado,entidad_federativa
0,1,Baja California
1,2,Baja California Sur
2,3,Campeche
3,4,Chiapas
4,5,Chihuahua


── dim_tiempo ──────────────────────────────


,Id_tiempo,anio
0,1,2013
1,2,2014
2,3,2015
3,4,2018
4,5,2019


── dim_fenomeno ────────────────────────────


,Id_fenomeno,tipo_fenomeno
0,1,Geologico (Hundimiento)
1,2,"Geologico (Sismo, Inestabilidad De Laderas Y T..."
2,3,"Geologico (Sismo, Inestabilidad De Laderas)"
3,4,"Geologico (Sismo, Vulcanismo, Hundimiento, Sub..."
4,5,"Geologico (Tsunamis, Karstificacion Y Cavidades)"


In [22]:
print(pd.read_sql('SELECT id_estado AS "Id_estado", entidad_federativa FROM desastres.dim_estado', engine))
print(pd.read_sql('SELECT id_tiempo AS "Id_tiempo", anio FROM desastres.dim_tiempo', engine))
print(pd.read_sql('SELECT id_fenomeno AS "Id_fenomeno", tipo_fenomeno FROM desastres.dim_fenomeno', engine))

    Id_estado   entidad_federativa
0           1      Baja California
1           2  Baja California Sur
2           3             Campeche
3           4              Chiapas
4           5            Chihuahua
5           6     Ciudad de México
6           7             Coahuila
7           8               Colima
8           9              Durango
9          10     Estado de México
10         11           Guanajuato
11         12             Guerrero
12         13              Hidalgo
13         14              Jalisco
14         15            Michoacán
15         16              Nayarit
16         17           Nuevo León
17         18               Oaxaca
18         19               Puebla
19         20         Quintana Roo
20         21      San Luis Potosí
21         22              Sinaloa
22         23               Sonora
23         24              Tabasco
24         25           Tamaulipas
25         26             Veracruz
26         27              Yucatán
27         28       

---
## 11. LOAD — `fact_riesgo`

Pasos:
1. Resolver surrogate keys (merge con las dims).
2. Cargar en **chunks de 2,000 filas** con barra de progreso.

> La barra de progreso (`tqdm.notebook`) es visible directamente en Jupyter.

In [23]:
def load_fact(fact: pd.DataFrame,
              dim_estado: pd.DataFrame,
              dim_tiempo: pd.DataFrame,
              dim_fenomeno: pd.DataFrame,
              engine,
              chunksize: int = 2000):

    # Resolver surrogate keys
    f = fact.merge(dim_estado,   on='entidad_federativa')
    f = f.merge(dim_tiempo,      on='anio')
    f = f.merge(dim_fenomeno,    on='tipo_fenomeno', how='left')

    fact_cols = [
        'Id_estado', 'Id_tiempo', 'Id_fenomeno',
        'total_desastres', 'total_emergencias',
        'municipios_afectados', 'poblacion_afectada',
        'inversion_prevencion',
    ]
    f = f[fact_cols].copy()

    # Renombrar a minúsculas para que coincidan con las columnas de Aurora
    f = f.rename(columns={
        'Id_estado':   'id_estado',
        'Id_tiempo':   'id_tiempo',
        'Id_fenomeno': 'id_fenomeno',
    })

    n_chunks = (len(f) + chunksize - 1) // chunksize
    for i in tqdm(range(n_chunks), desc='Cargando fact_riesgo', leave=True):
        chunk = f.iloc[i * chunksize:(i + 1) * chunksize]
        chunk.to_sql(
            'fact_riesgo', engine, schema='desastres',
            if_exists='append', index=False, method='multi',
            dtype={
                'id_estado':            Integer(),
                'id_tiempo':            Integer(),
                'id_fenomeno':          Integer(),
                'total_desastres':      Integer(),
                'total_emergencias':    Integer(),
                'municipios_afectados': Integer(),
                'poblacion_afectada':   Numeric(15, 2),
                'inversion_prevencion': Numeric(18, 2),
            },
        )
    print(f'✓ fact_riesgo cargada: {len(f):,} filas')


load_fact(fact, dim_estado, dim_tiempo, dim_fenomeno, engine)

Cargando fact_riesgo: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]

✓ fact_riesgo cargada: 545 filas


---
## 12. VALIDATE — Checks post-carga

Tres validaciones automáticas:
1. **Sin FK nulas** — `Id_estado` e `Id_tiempo` no pueden ser `NULL`.
2. **Sin valores negativos** en métricas numéricas.
3. **Resumen de totales** — muestra top 10 estado-año por desastres.

In [24]:
def validate(engine):

    # ── Check 1: FK nulas ──────────────────────────────────────────────────
    nulls = pd.read_sql(text("""
        SELECT COUNT(*) AS n FROM desastres.fact_riesgo
        WHERE Id_estado IS NULL OR Id_tiempo IS NULL
    """), engine).iloc[0, 0]
    assert nulls == 0, f'ERROR: {nulls} registros con FK nulas en fact_riesgo'
    print('✓ Check 1 PASSED — Sin FK nulas')

    # ── Check 2: valores negativos ────────────────────────────────────────
    negatives = pd.read_sql(text("""
        SELECT COUNT(*) AS n FROM desastres.fact_riesgo
        WHERE total_desastres < 0
           OR total_emergencias < 0
           OR poblacion_afectada < 0
           OR inversion_prevencion < 0
    """), engine).iloc[0, 0]
    assert negatives == 0, f'ERROR: {negatives} registros con valores negativos'
    print('✓ Check 2 PASSED — Sin valores negativos')

    # ── Check 3: resumen de totales ───────────────────────────────────────
    totales = pd.read_sql(text("""
        SELECT COUNT(*) AS registros_fact,
               SUM(total_desastres)   AS total_desastres,
               SUM(total_emergencias) AS total_emergencias,
               ROUND(SUM(poblacion_afectada)::numeric, 0) AS poblacion_total
        FROM desastres.fact_riesgo
    """), engine)
    print('\n── Resumen final fact_riesgo ──────────────────')
    display(totales)

    # ── Top 10 estado-año ─────────────────────────────────────────────────
    top10 = pd.read_sql(text("""
        SELECT de.entidad_federativa, dt.anio,
               SUM(fr.total_desastres)   AS desastres,
               SUM(fr.total_emergencias) AS emergencias,
               SUM(fr.poblacion_afectada)::bigint AS poblacion_afectada
        FROM desastres.fact_riesgo fr
        JOIN desastres.dim_estado de USING (Id_estado)
        JOIN desastres.dim_tiempo dt USING (Id_tiempo)
        GROUP BY de.entidad_federativa, dt.anio
        ORDER BY desastres DESC
        LIMIT 10
    """), engine)
    print('\n── Top 10 estado-año por desastres ────────────')
    display(top10)


validate(engine)

✓ Check 1 PASSED — Sin FK nulas
✓ Check 2 PASSED — Sin valores negativos

── Resumen final fact_riesgo ──────────────────


,registros_fact,total_desastres,total_emergencias,poblacion_total
0,545,610,950,15953070.0



── Top 10 estado-año por desastres ────────────


,entidad_federativa,anio,desastres,emergencias,poblacion_afectada
0,Chiapas,2020,50,70,1452505
1,Oaxaca,2020,40,110,261880
2,Chiapas,2022,30,10,165470
3,Veracruz,2020,30,50,390060
4,Chiapas,2019,25,50,1089450
5,Tabasco,2020,25,35,892590
6,Veracruz,2022,20,0,0
7,Durango,2020,15,35,165650
8,Veracruz,2023,15,5,30000
9,Veracruz,2019,15,20,52665


---
## 13. Cierre de conexión

Libera el connection pool de SQLAlchemy al finalizar el notebook.

In [ ]:
engine.dispose()
print('✓ ETL completado correctamente. Conexión cerrada.')